In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy pandas
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Trening kwantowego jądra (Quantum kernel training)

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*Szacowany czas użycia: poniżej jednej minuty na procesorze Eagle r3 (UWAGA: to tylko szacunek. Twój czas wykonania może się różnić.)*
## Tło
Ten samouczek pokazuje, jak zbudować wzorzec `Qiskit pattern` do obliczania wpisów macierzy kwantowego jądra używanej w klasyfikacji binarnej. Aby dowiedzieć się więcej o wzorcach `Qiskit patterns` i jak `Qiskit Serverless` może służyć do wdrażania ich w chmurze jako zarządzanego środowiska wykonawczego, odwiedź naszą [stronę dokumentacji dotyczącą IBM Quantum&reg; Platform](/guides/serverless).
## Wymagania
Przed rozpoczęciem tego samouczka upewnij się, że masz zainstalowane następujące elementy:
- Qiskit SDK v1.0 lub nowszy, z obsługą [wizualizacji](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 lub nowszy (`pip install qiskit-ibm-runtime`)
## Konfiguracja

In [ ]:
# General Imports and helper functions
import urllib.request

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.circuit.library import unitary_overlap
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

# Download the dataset (portable across platforms)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/qiskit-community/prototype-quantum-kernel-training/main/data/dataset_graph7.csv",
    "dataset_graph7.csv",
)


def visualize_counts(res_counts, num_qubits, num_shots):
    """Visualize the outputs from the Qiskit Sampler primitive."""
    zero_prob = res_counts.get(0, 0.0)
    top_10 = dict(
        sorted(res_counts.items(), key=lambda item: item[1], reverse=True)[
            :10
        ]
    )
    top_10.update({0: zero_prob})
    by_key = dict(sorted(top_10.items(), key=lambda item: item[0]))
    x_vals, y_vals = list(zip(*by_key.items()))
    x_vals = [bin(x_val)[2:].zfill(num_qubits) for x_val in x_vals]
    y_vals_prob = []
    for t in range(len(y_vals)):
        y_vals_prob.append(y_vals[t] / num_shots)
    y_vals = y_vals_prob
    plt.bar(x_vals, y_vals)
    plt.xticks(rotation=75)
    plt.title("Results of sampling")
    plt.xlabel("Measured bitstring")
    plt.ylabel("Probability")
    plt.show()


def get_training_data():
    """Read the training data."""
    df = pd.read_csv("dataset_graph7.csv", sep=",", header=None)
    training_data = df.values[:20, :]
    ind = np.argsort(training_data[:, -1])
    X_train = training_data[ind][:, :-1]

    return X_train

## Small-scale simulator example

In this section, we walk through the four steps of the Qiskit pattern on a seven-qubit instance of the labeling-cosets-with-error problem and evaluate a single kernel matrix entry using the `StatevectorSampler` primitive from Qiskit. A statevector simulator is exact (up to shot noise) and shows us the method end-to-end without consuming QPU time. We then repeat the same instance on real hardware in the hardware example section.

### Step 1: Map classical inputs to a quantum problem

*   Input: Training dataset.
*   Output: Abstract circuit for calculating a kernel matrix entry.

The binary classification problem we aim to solve here is referred to as "[_labeling cosets with error_](https://arxiv.org/abs/2105.03406)." The input training dataset contains a group structure, consisting of two cosets formed by a group and subgroup.
The group is taken to be $G = SU(2)^{\otimes n}$ for qubits, which is the special unitary group of
$2 \times 2$ matrices and has wide applicability in nature; e.g., the Standard Model of particle physics.
We take the (graph-stabilizer) subgroup $S_\text{graph} < G$ with $S_\text{graph} = \langle \{ X_i \otimes _{k:(k,i) \in \mathcal{E}} Z_k\} _{i \in \mathcal{V}} \} \rangle$ for a graph with edges $\mathcal{E}$ and vertices $\mathcal{V}$.
Note that the stabilizers fix a stabilizer state such that $D_s | \psi \rangle = | \psi \rangle,~ \forall s \in S_\text{graph}$.
Finally, we define two left-cosets $C_\pm = c_\pm S_\text{graph}$ by drawing two $c_\pm \in G$ at random.

For more details about the dataset and how it is generated, see [this notebook](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/docs/background/qkernels_and_data_w_group_structure.ipynb) from the [Quantum Kernel Training Toolkit](https://github.com/qiskit-community/prototype-quantum-kernel-training/tree/main).

We create the quantum circuit used to evaluate one entry in the kernel matrix.
The input data is used to determine the rotation angles for the circuit's parametrized gates.
For simplicity, we will use data samples `x1=14` and `x2=19`.

***Note: The dataset used in this tutorial can be downloaded [here](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/data/dataset_graph7.csv).***

In [2]:
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and set the data params for first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()
overlap_circ.draw("mpl", scale=0.6, style="iqp")

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif" alt="Output of the previous code cell" />

## Krok 1: Odwzorowanie klasycznych danych wejściowych na problem kwantowy
*   Dane wejściowe: Zbiór danych treningowych.
*   Dane wyjściowe: Abstrakcyjny Circuit do obliczania jednego wpisu macierzy jądra.

Utwórz Circuit kwantowy służący do obliczenia jednego wpisu macierzy jądra. Używamy danych wejściowych do określenia kątów obrotu dla sparametryzowanych bramek Circuit. Użyjemy próbek danych `x1=14` i `x2=19`.

***Uwaga: Zbiór danych użyty w tym samouczku można pobrać [tutaj](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/data/dataset_graph7.csv).***

In [3]:
sampler = StatevectorSampler()

# Execute and get counts
num_shots = 10_000
results = sampler.run([overlap_circ], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()

# Plot counts
visualize_counts(counts, num_qubits, num_shots)

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/step3-small-code-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif)

## Krok 2: Optymalizacja problemu pod kątem sprzętu kwantowego
*   Dane wejściowe: Abstrakcyjny Circuit, nieoptymalizowany pod kątem konkretnego Backend
*   Dane wyjściowe: Docelowy Circuit i obserwabla, zoptymalizowane pod wybrany QPU

Użyj funkcji `generate_preset_pass_manager` z Qiskit, aby określić procedurę optymalizacji Circuit w odniesieniu do QPU, na którym planujemy przeprowadzić eksperyment. Ustawiamy `optimization_level=3`, co oznacza, że użyjemy wstępnie skonfigurowanego menedżera przejść zapewniającego najwyższy poziom optymalizacji.

In [4]:
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (simulator): {kernel_matrix[x1, x2]}")

Fidelity (simulator): 0.8261


![Output of the previous code cell](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/49607b34-9723-493d-85da-bd97c1351104-0.avif)

## Krok 3: Wykonanie przy użyciu prymitywów Qiskit
*   Dane wejściowe: Docelowy Circuit
*   Dane wyjściowe: Rozkład quasi-prawdopodobieństwa

Użyj prymitywu `Sampler` z Qiskit Runtime, aby zrekonstruować rozkład quasi-prawdopodobieństwa stanów uzyskanych przez próbkowanie Circuit. W zadaniu generowania macierzy jądra szczególnie interesuje nas prawdopodobieństwo zmierzenia stanu |0>.

W tym demo uruchomimy obliczenia na QPU za pomocą prymitywów `qiskit-ibm-runtime`. Aby uruchomić je na prymitywach opartych na wektorze stanu `qiskit`, zastąp blok kodu używający prymitywów Qiskit IBM&reg; Runtime zakomentowanym blokiem.

In [5]:
# ------------------------------ Step 1 ------------------------------
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and set the data params for first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()

# ------------------------------ Step 2 ------------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=overlap_circ.num_qubits
# )
backend = service.backend("ibm_pittsburgh")
print(f"Using backend: {backend.name}")
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
overlap_ibm = pm.run(overlap_circ)

# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_QKT"]

num_shots = 10_000
results = sampler.run([overlap_ibm], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()
visualize_counts(counts, num_qubits, num_shots)

# ------------------------------ Step 4 ------------------------------
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (hardware): {kernel_matrix[x1, x2]}")

Using backend: ibm_pittsburgh


<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/d2f4f6cf-067e-4d53-aa04-7ca9c803d3e1-1.avif" alt="Output of the previous code cell" />

Fidelity (hardware): 0.7517


![Output of the previous code cell](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/d2f4f6cf-067e-4d53-aa04-7ca9c803d3e1-0.avif)

## Krok 4: Post-processing i zwrócenie wyniku w żądanym formacie klasycznym

*   Dane wejściowe: Rozkład prawdopodobieństwa
*   Dane wyjściowe: Pojedynczy element macierzy jądra

Oblicz prawdopodobieństwo zmierzenia |0> na Circuit nakładkowym i wypełnij macierz jądra na pozycji odpowiadającej próbkom reprezentowanym przez ten konkretny Circuit (wiersz 15, kolumna 20). Na tej wizualizacji ciemniejszy czerwony oznacza wierności bliższe 1,0. Aby wypełnić całą macierz jądra, musimy przeprowadzić eksperyment kwantowy dla każdego wpisu.